In [1]:
%%capture --no-stderr
%pip install --upgrade --force-reinstall langgraph
%pip install --quiet -U langchain_openai langchain_core
%pip install requests

curl https://llmexecution.api.intuit.com/v3/models

## Generate an Access Token by running the following curl command either in the terminal or postman/intuit api client

### Note - First replace the app_secret in the command with secret shared in TinCan







```
curl --location 'https://identityinternal.api.intuit.com/signin/graphql' --header 'Authorization: Intuit_IAM_Authentication intuit_appid="Intuit.aifabric.genos.agenticaitrainingclient",intuit_app_secret=prdFtgimomcXJHvKvJo80831nB6FINieM2HKkGwO' --header 'Content-Type: application/json' --data-raw '{"query":"mutation identityTestSignInWithPassword($input: Identity_TestSignInWithPasswordInput!) {\n identityTestSignInWithPassword(input: $input) {\n accessToken\n legacyAuthId\n}\n}","variables":{"input":{"username":"iamtestpass_616829375333307","password":"Intuit01-","tenantId":"50000003","intent":{"appGroup":"QBO","assetAlias":"Intuit.aifabric.genos.agenticaitrainingclient"}}}}'
```




## Generate the PrivateAuth+ headers for calling the LLM
### Note - First replace the intuit_app_secret and intuit_token fields in the AUTHN_STRING below with the app_secret shared in TinCan and accessToken fromt the previous step

In [2]:
# Intuit PrivateAuth+ headers
# Use the Access Token from the previous cell in the intuit_token_type field
AUTHN_STRING = "Intuit_IAM_Authentication " \
               "intuit_appid='Intuit.aifabric.genos.agenticaitrainingclient'," \
               "intuit_app_secret='prdFtgimomcXJHvKvJo80831nB6FINieM2HKkGwO'," \
               "intuit_token_type='IAM-Ticket'," \
               f"intuit_userid=4621097838312242983," \
               f"intuit_token=V1-32-B0iteu1r8llg2nvbmq2f9d"

INTUIT_AUTHN_HEADERS = {
    "Authorization": AUTHN_STRING,
    "intuit_experience_id": "c1392e45-9ef3-4d87-9e6f-3dd45294e545", # GenOS experience_id
    "intuit_originating_assetalias": "Intuit.aifabric.genos.agenticaitrainingclient"
}

## Define tools and bind to LLM

In [3]:
from langchain_openai import ChatOpenAI
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second intfrom langchain_openai import ChatOpenAI
    """
    return a * b

# This will be a tool
def add(a: int, b: int) -> int:
    """Adds a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b

def divide(a: int, b: int) -> float:
    """Divide a and b.

    Args:
        a: first int
        b: second int
    """
    return a / b

tools = [add, multiply, divide]

llm = ChatOpenAI(
    api_key="anything",  # No OP as we will use Intuit AuthN headers
    model="xxxx",  # No OP as model id is in path param
    base_url="https://llmexecution.api.intuit.com/v3/gpt-4o-2024-05-13/",
    temperature=0,
    max_tokens=4096,
    extra_headers={
        **INTUIT_AUTHN_HEADERS
    })

llm_with_tools = llm.bind_tools(tools)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3473: UserWarning: WARNING! extra_headers is not default parameter.
                extra_headers was transferred to model_kwargs.
                Please confirm that extra_headers is what you intended.
  if (await self.run_code(code, result,  async_=asy)):


## Run LLM using langchain without any tool

In [4]:
message = [
    ('system', 'please assist user in calculating'),
    ('user', 'what is 3747345/347 Think step by step.')
]

results = llm.invoke(message)
print(results.text)

To calculate \( \frac{3747345}{347} \), we can perform the division step by step.

1. First, we set up the division problem: \( 3747345 \div 347 \).

2. We start by seeing how many times 347 fits into the first few digits of 3747345:
   - 347 goes into 3747 (the first four digits) approximately 10 times because \( 347 \times 10 = 3470 \).
   - Subtract 3470 from 3747: \( 3747 - 3470 = 277 \).

3. Bring down the next digit (3), making it 2773.
   - 347 goes into 2773 approximately 8 times because \( 347 \times 8 = 2776 \) is too large, so we use 7 times: \( 347 \times 7 = 2429 \).
   - Subtract 2429 from 2773: \( 2773 - 2429 = 344 \).

4. Bring down the next digit (4), making it 3444.
   - 347 goes into 3444 approximately 9 times because \( 347 \times 9 = 3123 \).
   - Subtract 3123 from 3444: \( 3444 - 3123 = 321 \).

5. Bring down the next digit (5), making it 3215.
   - 347 goes into 3215 approximately 9 times because \( 347 \times 9 = 3123 \).
   - Subtract 3123 from 3215: \( 3215 -

## Tool calling under the hood

In [5]:
results = llm.invoke([
    ('system', 'Please assist the user in calculating'),
    ('user', 'What is 234293874 / 342, use a tool to calculate this, do not perform the calculation manually, by returning JSON in the following format: { "function_call": "divide", "numerator": 5, "divisor": 2}')
])

print(results.content)

{
  "function_call": "divide",
  "numerator": 234293874,
  "divisor": 342
}


In [6]:
results.content.split("```json")

['{\n  "function_call": "divide",\n  "numerator": 234293874,\n  "divisor": 342\n}']

In [7]:
json_results = results.content.split("```json")[0].split("```")[0]
import json
print(json.loads(json_results))
tool_call = json.loads(json_results)

{'function_call': 'divide', 'numerator': 234293874, 'divisor': 342}


In [8]:
tool_call

{'function_call': 'divide', 'numerator': 234293874, 'divisor': 342}

In [9]:
def divide(numerator,divisor):
  return numerator / divisor
tools = {
    "divide": divide
}

result = tools[tool_call["function_call"]](tool_call["numerator"], tool_call["divisor"])
print(result)

685069.8070175438


In [10]:
results = llm.invoke([
    ('system', 'Please assist the user in calculating'),
    ('user', 'What is 234293874 / 342, use a tool to calculate this, do not perform the calculation manually, by returning JSON in the following format: { "function_call": "divide", "numerator": 5, "divisor": 2}'),
    ('assistant', results.content),
    ('user', "The tool call's result is: " + str(result))
])
print(results)

content='The result of dividing 234293874 by 342 is approximately 685069.8070175438.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 122, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'text_tokens': None, 'image_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_ee1d74bde0', 'id': 'chatcmpl-CbXdfOufMQkqn45X1mxFX5OBpAVsR', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--ba43714a-18dd-4aa8-9a4c-447375bfb1a1-0' usage_metadata={'input_tokens': 122, 'output_tokens': 23, 'total_tokens': 145, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## Run LLM using langchain with tools

In [ ]:
message = [
    ('system', 'please assist user in calculating'),
    ('user', 'what is 3747345/347 Think step by step.')
]

results = llm_with_tools.invoke(message)
print(results)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 138, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'text_tokens': None, 'image_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_ee1d74bde0', 'id': 'chatcmpl-CbCIftknbULp1drJVx1FLNTqru4Me', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--fc927c94-ad3f-4b77-acc2-e8615bfb56f8-0' tool_calls=[{'name': 'divide', 'args': {'a': 3747345, 'b': 347}, 'id': 'call_1HuKt7cI7c68R4sraz7toE2j', 'type': 'tool_call'}] usage_metadata={'input_tokens': 138, 'output_tokens': 20, 'total_tokens': 158, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## Add assistant

In [11]:
from langgraph.graph import MessagesState
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# System message
sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing arithmetic on a set of inputs.")

# Node
def assistant(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

## Build the graph

In [12]:
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode
from IPython.display import Image, display

# Graph
builder = StateGraph(MessagesState)

# Define nodes: these do the work
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition,
)
builder.add_edge("tools", "assistant")
react_graph = builder.compile(debug=True)

# Show
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

AttributeError: 'function' object has no attribute 'name'

## Invoke the agent

In [ ]:
## tracer goes here
messages = [HumanMessage(content="Add 389573 and 4439374. Then divide the whole thing by 6")]
messages = react_graph.invoke({"messages": messages})
for m in messages['messages']:
    m.pretty_print()

[values] {'messages': [HumanMessage(content='Add 389573 and 4439374. Then divide the whole thing by 6', additional_kwargs={}, response_metadata={}, id='ee7b3c54-ff75-47e6-a8f6-466b45ac930b')]}
[updates] {'assistant': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 152, 'total_tokens': 208, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'text_tokens': None, 'image_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_f353efaf53', 'id': 'chatcmpl-CbCItFIa7FmcQaILR6TcEbxIfZzyI', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--fa2884d3-96ef-4e38-bdab-989ab6ea9a35-0', tool_calls=[{'name': 'add', 'args': {'a': 389573, 'b': 4439374}, 'id': 'call_Xn7e4yb

## Using state

In [ ]:
## tracer goes here
config: R
messages = [HumanMessage(content="Add 389573 and 4439374. Then divide the whole thing by 6")]
messages = react_graph.invoke({"messages": messages})
for m in messages['messages']:
    m.pretty_print()